In [9]:
#!/usr/bin/env python3
"""
train.py
Train a fake-news classifier and save vectorizer + model artifacts.
"""

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from preprocess import clean_text  # your cleaning function

# Where models/artifacts will be stored
MODEL_DIR = "model_artifacts"
os.makedirs(MODEL_DIR, exist_ok=True)


def _normalize_label(v):
    """
    Normalize many possible label representations to 0/1.
    Returns np.nan if the label is unrecognized.
    """
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ("real", "1", "true", "t", "yes", "y"):
        return 1
    if s in ("fake", "0", "false", "f", "no", "n"):
        return 0
    return np.nan


def load_dataset(path="news_dataset.csv"):
    """
    Load CSV, ensure columns exist, clean text, normalize labels.
    Expects at least columns: 'text' and 'label'.
    """
    df = pd.read_csv(path)

    # Ensure text present and string type
    df = df.dropna(subset=["text"])
    df["text"] = df["text"].astype(str)

    # Ensure label column exists
    if "label" not in df.columns:
        raise KeyError("Input CSV must contain a 'label' column.")

    # Normalize label values (handle REAL/FAKE, True/False, 1/0, etc.)
    df["label"] = df["label"].apply(_normalize_label)

    # Drop rows where label could not be mapped
    df = df.dropna(subset=["label"])

    # Final label type = integer 0/1
    df["label"] = df["label"].astype(int)

    return df


def preprocess_and_train(path="news_dataset.csv"):
    """
    Full pipeline: load, preprocess, vectorize, train (GridSearch), evaluate, save.
    """
    df = load_dataset(path)

    # Create cleaned text column
    df["clean_text"] = df["text"].apply(clean_text)

    # Features & labels
    X = df["clean_text"]
    y = df["label"]

    # Train/test split with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # TF-IDF vectorizer
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    print("Vectorized shapes:", X_train_tfidf.shape, X_test_tfidf.shape)

    # Grid-search over regularization strength for LogisticRegression
    param_grid = {"C": [0.1, 1, 3, 10]}
    lr = LogisticRegression(max_iter=5000, class_weight="balanced", solver="liblinear")
    grid = GridSearchCV(lr, param_grid, cv=3, scoring="f1", n_jobs=-1)
    grid.fit(X_train_tfidf, y_train)
    best = grid.best_estimator_
    print("Best params:", grid.best_params_)

    # Evaluate on test set
    y_pred = best.predict(X_test_tfidf)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification report:\n", classification_report(y_test, y_pred))

    # Save artifacts
    joblib.dump(vectorizer, os.path.join(MODEL_DIR, "vectorizer.joblib"))
    joblib.dump(best, os.path.join(MODEL_DIR, "model.joblib"))
    print("Saved model and vectorizer to", MODEL_DIR)


if __name__ == "__main__":
    preprocess_and_train("news_dataset.csv")

Vectorized shapes: (2976, 10000) (745, 10000)
Best params: {'C': 10}
Accuracy: 0.9946308724832215
Classification report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       375
           1       1.00      0.99      0.99       370

    accuracy                           0.99       745
   macro avg       0.99      0.99      0.99       745
weighted avg       0.99      0.99      0.99       745

Saved model and vectorizer to model_artifacts
